# Classical ML Training Notebook

This notebook trains a simple ridge-regression baseline on the CSV leaderboard file. It treats `Score` as the default target and drops `Rank` and `TeamId` from features.

In [ ]:
from pathlib import Path

import numpy as np

from tabular_utils import (
    TabularPreprocessor,
    extract_target,
    infer_target_column,
    load_csv_rows,
    regression_metrics,
    train_test_indices,
)

In [ ]:
CSV_PATH = Path("orbit-wars-publicleaderboard-2026-05-05T15:36:33.csv")
TARGET_COLUMN = None  # set to a specific column name if needed
DROP_COLUMNS = ["Rank", "TeamId"]
TEST_SIZE = 0.25
SEED = 42
RIDGE = 1e-3

In [ ]:
def ridge_fit(features: np.ndarray, target: np.ndarray, ridge: float = 1e-3) -> np.ndarray:
    design = np.c_[np.ones((features.shape[0], 1)), features]
    identity = np.eye(design.shape[1])
    identity[0, 0] = 0.0
    return np.linalg.solve(design.T @ design + ridge * identity, design.T @ target)


def ridge_predict(features: np.ndarray, weights: np.ndarray) -> np.ndarray:
    design = np.c_[np.ones((features.shape[0], 1)), features]
    return design @ weights

In [ ]:
columns, rows = load_csv_rows(str(CSV_PATH))
if not rows:
    raise ValueError(f"CSV file is empty: {CSV_PATH}")

target_column = TARGET_COLUMN or infer_target_column(columns)
if target_column not in columns:
    raise ValueError(f"Target column '{target_column}' not found in {CSV_PATH}.")

drop_columns = {column for column in DROP_COLUMNS if column in columns and column != target_column}
feature_columns = [column for column in columns if column != target_column and column not in drop_columns]
if not feature_columns:
    raise ValueError("No feature columns remain after dropping the target and excluded columns.")

train_idx, test_idx = train_test_indices(len(rows), test_size=TEST_SIZE, seed=SEED)
train_rows = [rows[index] for index in train_idx]
test_rows = [rows[index] for index in test_idx]

preprocessor = TabularPreprocessor()
x_train = preprocessor.fit_transform(train_rows, feature_columns)
x_test = preprocessor.transform(test_rows)
y_train = extract_target(train_rows, target_column)
y_test = extract_target(test_rows, target_column)

weights = ridge_fit(x_train, y_train, ridge=RIDGE)
predictions = ridge_predict(x_test, weights)
metrics = regression_metrics(y_test, predictions)

print(f"CSV: {CSV_PATH}")
print(f"Target: {target_column}")
print(f"Features: {', '.join(feature_columns)}")
print(f"Train rows: {len(train_rows)} | Test rows: {len(test_rows)}")
print("Metrics:")
print(f"  MAE:  {metrics['mae']:.4f}")
print(f"  RMSE: {metrics['rmse']:.4f}")
print(f"  R2:   {metrics['r2']:.4f}")
print("Predictions vs actual:")
for actual, predicted in zip(y_test[:10], predictions[:10]):
    print(f"  actual={actual:.3f} predicted={predicted:.3f}")